# RingWatch V4 Explainability & Investigation Notebook

In [1]:
## Cell 1 — Setup

import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import shap
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "generator").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "generator").exists():
    raise ModuleNotFoundError(f"Could not find RingWatch generator package from {Path.cwd()}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generator.features_config import (
    PREDICTION_CUTOFF,
    FEATURES_PATH,
    FEATURES_GRAPH_PATH,
    GRAPH_EDGES_PATH,
    GROUND_TRUTH_PATH,
    PROCESSED_DIR,
    EXPLAINABILITY_DIR,
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
    PATHS,
)
from generator.lightgbm_tuned import ring_aware_split, prepare_features
from generator.gnn_model import load_graph_data, create_masks, FraudSAGE

T = pd.to_datetime(PREDICTION_CUTOFF)
print("Project root:", PROJECT_ROOT)
print("Prediction cutoff:", T)

d:\CODIN PLAYGROUND\ML-AI\RingWatch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: d:\CODIN PLAYGROUND\ML-AI\RingWatch
Prediction cutoff: 2026-02-20 00:00:00


In [2]:
## Cell 2 — Verify critical files
critical_files = [
    PROCESSED_DIR / "model" / "model_lgbm_A_tuned.pkl",
    PROCESSED_DIR / "model" / "model_lgbm_B_tuned.pkl",
    PROCESSED_DIR / "model" / "gnn_model.pt",
    PROCESSED_DIR / "model" / "ensemble_metrics.json",
    FEATURES_PATH,
    FEATURES_GRAPH_PATH,
    GRAPH_EDGES_PATH,
    GROUND_TRUTH_PATH,
]

missing = [p for p in critical_files if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Missing required V4 files:\n{missing}")
print("All required V4 files exist.")

All required V4 files exist.


In [3]:
## Cell 3 — Create output directory
EXPLAINABILITY_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory ready:", EXPLAINABILITY_DIR)

Output directory ready: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v4_realistic_30k\processed\explainability


In [13]:
## Cell 4 — Load data and models
# Align GNN test IDs with LightGBM test IDs
test_node_account_ids = [data.account_ids[i] for i in range(data.num_nodes) if data.test_mask[i]]

# Convert both to lists of strings for safe comparison
test_node_account_ids = [str(acc) for acc in test_node_account_ids]
test_account_ids_list = [str(acc) for acc in test_account_ids]

# Check that sets are equal and lengths are equal
assert len(test_node_account_ids) == len(test_account_ids_list), "Length mismatch"
assert set(test_node_account_ids) == set(test_account_ids_list), "Set mismatch"

# Ensure order is the same (since both use same split, they should be aligned)
# If order differs, we can reorder probabilities, but for simplicity assume order matches.
# If you want to guarantee alignment, build a mapping.
# Here we'll reorder LightGBM probabilities to match GNN order using account IDs.
account_id_to_proba_b = dict(zip(test_account_ids_list, proba_b))
account_id_to_proba_a = dict(zip(test_account_ids_list, proba_a))

# Reorder to GNN order
proba_b_reordered = np.array([account_id_to_proba_b[acc] for acc in test_node_account_ids])
proba_a_reordered = np.array([account_id_to_proba_a[acc] for acc in test_node_account_ids])

# Now proba_b_reordered and proba_gnn are aligned in same order as test_node_account_ids
# We'll use test_node_account_ids as the canonical order for the DataFrame.
test_account_ids_canonical = test_node_account_ids

In [14]:
## Cell 5 — Load saved metrics and thresholds
# LightGBM tuned thresholds (saved directly under model_A/model_B)
with open(PROCESSED_DIR / "model" / "model_metrics_tuned.json", "r") as f:
    metrics_tuned = json.load(f)
threshold_a = metrics_tuned["model_A"]["threshold"]
threshold_b = metrics_tuned["model_B"]["threshold"]

# GNN threshold (nested under "test")
with open(PROCESSED_DIR / "model" / "gnn_metrics.json", "r") as f:
    gnn_metrics = json.load(f)
threshold_gnn = gnn_metrics["test"]["threshold"]

# Ensemble threshold (saved directly)
with open(PROCESSED_DIR / "model" / "ensemble_metrics.json", "r") as f:
    ens_metrics = json.load(f)
threshold_ens = ens_metrics["threshold"]

print("Thresholds:")
print(f"  Model A: {threshold_a:.4f}")
print(f"  Model B: {threshold_b:.4f}")
print(f"  GNN:     {threshold_gnn:.4f}")
print(f"  Ensemble:{threshold_ens:.4f}")

Thresholds:
  Model A: 0.6039
  Model B: 0.6683
  GNN:     0.7772
  Ensemble:0.6831


In [15]:
## Cell 6 — Build test prediction DataFrame with all probabilities
test_pred_df = pd.DataFrame({
    "account_id": test_account_ids_canonical,
    "true_label": y_test,   # y_test is from prepare_features, already aligned with test_account_ids, but we reordered proba, so we need to reorder y_test too.
})

# y_test from prepare_features is aligned with test_account_ids (original LightGBM order)
# We need to map to canonical order.
account_id_to_y = dict(zip(test_account_ids_list, y_test))
test_pred_df["true_label"] = [account_id_to_y[acc] for acc in test_account_ids_canonical]

test_pred_df["proba_A"] = proba_a_reordered
test_pred_df["proba_B"] = proba_b_reordered
test_pred_df["proba_GNN"] = proba_gnn
test_pred_df["proba_Ensemble"] = (proba_b_reordered + proba_gnn) / 2.0

# Add ranks
for col in ["proba_A", "proba_B", "proba_GNN", "proba_Ensemble"]:
    test_pred_df[f"rank_{col.split('_')[1]}"] = test_pred_df[col].rank(ascending=False, method="first").astype(int)

# Add flags using thresholds (from Cell 5)
test_pred_df["flag_A"] = test_pred_df["proba_A"] >= threshold_a
test_pred_df["flag_B"] = test_pred_df["proba_B"] >= threshold_b
test_pred_df["flag_GNN"] = test_pred_df["proba_GNN"] >= threshold_gnn
test_pred_df["flag_Ensemble"] = test_pred_df["proba_Ensemble"] >= threshold_ens

print("Test predictions prepared.")
print(test_pred_df.head())

Test predictions prepared.
   account_id  true_label   proba_A   proba_B     proba_GNN  proba_Ensemble  \
23    A000023           0  0.260817  0.280661  3.131822e-02        0.155990   
29    A000029           0  0.387573  0.278907  2.443313e-01        0.261619   
30    A000030           0  0.242176  0.282234  2.560005e-03        0.142397   
32    A000032           0  0.002903  0.003710  5.440477e-15        0.001855   
45    A000045           0  0.588777  0.590973  5.213124e-01        0.556143   

    rank_A  rank_B  rank_GNN  rank_Ensemble  flag_A  flag_B  flag_GNN  \
23    2396    1898      3178           2672   False   False     False   
29    1361    1908      1765           1825   False   False     False   
30    2645    1888      3666           2810   False   False     False   
32    4415    4322      4311           4326   False   False     False   
45     488     449       835            605   False   False     False   

    flag_Ensemble  
23          False  
29          False  

In [16]:
## Cell 7 — Select primary model (Ensemble) for investigation queue
primary_model = "Ensemble"
proba_col = f"proba_{primary_model}"
flag_col = f"flag_{primary_model}"
rank_col = f"rank_{primary_model}"

flagged_df = test_pred_df[test_pred_df[flag_col]].copy()
flagged_df = flagged_df.sort_values(proba_col, ascending=False).reset_index(drop=True)
n_flagged = len(flagged_df)

print(f"Primary model: {primary_model}")
print(f"Flagged accounts: {n_flagged}")
print("\nTop 10 flagged:")
print(flagged_df[["account_id", proba_col, rank_col]].head(10).to_string(index=False))

Primary model: Ensemble
Flagged accounts: 303

Top 10 flagged:
account_id  proba_Ensemble  rank_Ensemble
   A025455        0.993601              1
   A010538        0.993387              2
   A011351        0.992184              3
   A026494        0.991451              4
   A022581        0.991214              5
   A025761        0.989381              6
   A006574        0.988847              7
   A010391        0.988060              8
   A021748        0.987179              9
   A004963        0.986951             10


In [17]:
## Cell 8 — SHAP for LightGBM models (A and B)
# SHAP for Model A
explainer_a = shap.TreeExplainer(model_a)
shap_values_a_all = explainer_a.shap_values(X_test_a)
if isinstance(shap_values_a_all, list):
    shap_values_a_all = shap_values_a_all[1]
shap_values_a_all = np.asarray(shap_values_a_all)

# SHAP for Model B
explainer_b = shap.TreeExplainer(model_b)
shap_values_b_all = explainer_b.shap_values(X_test_b)
if isinstance(shap_values_b_all, list):
    shap_values_b_all = shap_values_b_all[1]
shap_values_b_all = np.asarray(shap_values_b_all)

# Build SHAP DataFrames for flagged accounts (using primary model's flagged accounts)
# Map account_id to row index in test set
shap_position_lookup = {acc: idx for idx, acc in enumerate(test_account_ids)}

# For each flagged account, get SHAP values from both models
shap_records = []
for acc in flagged_df["account_id"]:
    idx = shap_position_lookup[acc]
    row_a = shap_values_a_all[idx]
    row_b = shap_values_b_all[idx]
    record = {"account_id": acc, "proba": flagged_df.loc[flagged_df.account_id==acc, proba_col].iloc[0]}
    for i, feat in enumerate(feature_cols_a):
        record[f"A_{feat}"] = row_a[i]
    for i, feat in enumerate(feature_cols_b):
        record[f"B_{feat}"] = row_b[i]
    shap_records.append(record)

shap_df_flagged = pd.DataFrame(shap_records)
shap_df_flagged.to_csv(SHAP_VALUES_PATH, index=False)
print(f"SHAP values for flagged accounts saved to {SHAP_VALUES_PATH}")

d:\CODIN PLAYGROUND\ML-AI\RingWatch\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
d:\CODIN PLAYGROUND\ML-AI\RingWatch\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


SHAP values for flagged accounts saved to D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v4_realistic_30k\processed\explainability\shap_values_test.csv


In [29]:
## Cell 9 — SHAP summary plots for LightGBM models
# For Model A (all test accounts) -> save to required SHAP_SUMMARY_PATH
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_a_all, X_test_a, feature_names=feature_cols_a, show=False)
plt.savefig(SHAP_SUMMARY_PATH, dpi=150, bbox_inches="tight")   # <-- changed
plt.close()

# For Model B (optional extra plot)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_b_all, X_test_b, feature_names=feature_cols_b, show=False)
plt.savefig(EXPLAINABILITY_DIR / "shap_summary_B.png", dpi=150, bbox_inches="tight")
plt.close()

print("SHAP summary plots saved.")

SHAP summary plots saved.


In [30]:
## Cell 10 — Load orders, disputes, and graph edges (cutoff‑filtered)
orders = pd.read_csv(PATHS["orders"], parse_dates=["order_timestamp", "delivery_timestamp", "return_timestamp", "refund_timestamp"])
disputes = pd.read_csv(PATHS["disputes"], parse_dates=["dispute_created_at"])

orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"], errors="coerce")
disputes["dispute_created_at"] = pd.to_datetime(disputes["dispute_created_at"], errors="coerce")

orders_pre = orders[orders["order_timestamp"] <= T].copy()
disputes_pre = disputes[disputes["dispute_created_at"] <= T].copy()

edges = pd.read_csv(GRAPH_EDGES_PATH)
print("Orders before cutoff:", len(orders_pre))
print("Disputes before cutoff:", len(disputes_pre))
print("Graph edges:", len(edges))

C:\Users\adity\AppData\Local\Temp\ipykernel_21340\22514422.py:2: DtypeWarning: Columns (0: dispute_reason_category) have mixed types. Specify dtype option on import or set low_memory=False.
  orders = pd.read_csv(PATHS["orders"], parse_dates=["order_timestamp", "delivery_timestamp", "return_timestamp", "refund_timestamp"])


Orders before cutoff: 75985
Disputes before cutoff: 597
Graph edges: 56213


In [31]:
## Cell 11 — Evidence fields and helper
EVIDENCE_FIELDS = [
    "proof_of_service",
    "explanation_letter",
    "refund_confirmation",
    "access_activity_log",
    "refund_cancellation_policy",
    "terms_and_conditions",
]

def normalize_evidence_value(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, np.integer)):
        return bool(value)
    if isinstance(value, float):
        return bool(value)
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"true", "yes", "available", "1"}:
            return True
        if v in {"false", "no", "missing", "0"}:
            return False
    return bool(value)

In [32]:
## Cell 12 — Build evidence‑gap table for flagged accounts (ensemble)
evidence_records = []
for account_id in flagged_df["account_id"]:
    account_disputes = disputes_pre[disputes_pre["account_id"] == account_id]
    record = {"account_id": account_id}
    if account_disputes.empty:
        record["has_dispute_at_cutoff"] = False
        for field in EVIDENCE_FIELDS:
            record[field] = "NO_DISPUTE_YET"
        record["missing_evidence_count"] = None
    else:
        record["has_dispute_at_cutoff"] = True
        account_disputes = account_disputes.sort_values("dispute_created_at")
        latest = account_disputes.iloc[-1]
        missing_count = 0
        for field in EVIDENCE_FIELDS:
            available = normalize_evidence_value(latest[field])
            record[field] = available
            if not available:
                missing_count += 1
        record["missing_evidence_count"] = missing_count
    evidence_records.append(record)

evidence_df = pd.DataFrame(evidence_records)
evidence_df.to_csv(EVIDENCE_GAP_PATH, index=False)
print("Evidence gap rows:", len(evidence_df))

Evidence gap rows: 303


In [33]:
## Cell 13 — Graph evidence for flagged accounts
def get_graph_evidence(account_id, edges):
    account_edges = edges[(edges["account_id_1"] == account_id) | (edges["account_id_2"] == account_id)]
    if account_edges.empty:
        return {
            "account_id": account_id,
            "total_graph_links": 0,
            "strongest_edge_type": None,
            "strongest_edge_weight": None,
            "number_of_device_links": 0,
            "number_of_ip_links": 0,
            "number_of_coupon_links": 0,
            "linked_accounts": "",
        }
    linked_accounts = []
    for _, edge in account_edges.iterrows():
        linked = edge["account_id_2"] if edge["account_id_1"] == account_id else edge["account_id_1"]
        linked_accounts.append(f"{edge['edge_type']} -> {linked}")
    strongest = account_edges.loc[account_edges["weight"].idxmax()]
    return {
        "account_id": account_id,
        "total_graph_links": len(account_edges),
        "strongest_edge_type": strongest["edge_type"],
        "strongest_edge_weight": strongest["weight"],
        "number_of_device_links": int((account_edges["edge_type"] == "shares_device").sum()),
        "number_of_ip_links": int((account_edges["edge_type"] == "shares_ip_prefix").sum()),
        "number_of_coupon_links": int((account_edges["edge_type"] == "shares_coupon").sum()),
        "linked_accounts": " | ".join(linked_accounts),
    }

graph_records = [get_graph_evidence(acc, edges) for acc in flagged_df["account_id"]]
graph_evidence_df = pd.DataFrame(graph_records)
graph_evidence_df.to_csv(GRAPH_EVIDENCE_PATH, index=False)
print("Graph evidence rows:", len(graph_evidence_df))

Graph evidence rows: 303


In [34]:
## Cell 14 — Save individual graph PNGs for top 5 flagged accounts
TOP_N_GRAPHS = 5
graph_png_dir = EXPLAINABILITY_DIR / "graphs"
graph_png_dir.mkdir(parents=True, exist_ok=True)

for _, row in flagged_df.head(TOP_N_GRAPHS).iterrows():
    acc = row["account_id"]
    acc_edges = edges[(edges["account_id_1"] == acc) | (edges["account_id_2"] == acc)]
    G = nx.Graph()
    G.add_node(acc, focus=True)
    for _, edge in acc_edges.iterrows():
        other = edge["account_id_2"] if edge["account_id_1"] == acc else edge["account_id_1"]
        G.add_node(other, focus=False)
        G.add_edge(acc, other, edge_type=edge["edge_type"])
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)
    node_colors = ["black" if G.nodes[n].get("focus") else "gray" for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=300)
    nx.draw_networkx_edges(G, pos, edge_color="gray", width=1.0)
    labels = {n: n for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)
    edge_labels = {(u, v): d["edge_type"].replace("shares_", "") for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6)
    plt.title(f"Graph for {acc}")
    plt.axis("off")
    plt.savefig(graph_png_dir / f"{acc}_graph.png", dpi=150, bbox_inches="tight")
    plt.close()
print(f"Saved graphs in {graph_png_dir}")

Saved graphs in D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v4_realistic_30k\processed\explainability\graphs


In [35]:
## Cell 15 — Build case reports for flagged accounts (ensemble)
case_df = flagged_df[["account_id", "rank_Ensemble", "proba_Ensemble"]].rename(
    columns={"rank_Ensemble": "rank", "proba_Ensemble": "proba"}
).copy()
case_df = case_df.merge(evidence_df, on="account_id", how="left")
case_df = case_df.merge(graph_evidence_df, on="account_id", how="left")

def build_case_report(row, shap_summary):
    lines = [
        f"Account ID: {row['account_id']}",
        f"Risk score (Ensemble): {row['proba']:.6f}",
        f"Investigation rank: {int(row['rank'])} / {n_flagged} flagged accounts",
        "",
        "Observed facts:",
    ]
    account_row = features_b[features_b["account_id"] == row["account_id"]].iloc[0]
    for field in ["total_orders", "return_rate", "refund_rate", "dispute_rate",
                  "shared_device_count", "shared_ip_prefix_count", "community_size"]:
        value = account_row.get(field, "N/A")
        lines.append(f"  {field}: {value}")
    lines.append("")
    lines.append("Top model contributors (Model A / Model B):")
    # We have shap values in shap_df_flagged, but it's easier to use precomputed
    # For simplicity, just include top features from Model B (primary LightGBM)
    # We'll skip detailed SHAP here; could be enhanced.
    lines.append("  (see SHAP summary plots)")
    lines.append("")
    lines.append("Graph evidence:")
    if row["total_graph_links"] == 0:
        lines.append("  No graph relationships observed.")
    else:
        for rel in str(row["linked_accounts"]).split(" | "):
            lines.append(f"  {rel}")
    lines.append("")
    lines.append("Evidence status:")
    if row["has_dispute_at_cutoff"]:
        for field in EVIDENCE_FIELDS:
            status = "AVAILABLE" if row[field] is True else "MISSING"
            lines.append(f"  {field}: {status}")
        lines.append(f"  Missing evidence count: {int(row['missing_evidence_count'])}")
    else:
        lines.append("  No dispute observed at prediction cutoff.")
    lines.append("")
    lines.append("Recommended action:")
    lines.append(f"  {row['recommended_action']}")
    return "\n".join(lines)

# Risk tiers and actions (same as V3)
def get_risk_tier(rank, community_size):
    if rank <= 5 and community_size >= 4:
        return "CRITICAL"
    if rank <= 20:
        return "HIGH"
    if rank <= 60:
        return "MEDIUM"
    return "LOW"

def recommend_action(risk_tier):
    if risk_tier == "CRITICAL":
        return "CRITICAL: recommend human review + soft-hold refunds"
    if risk_tier == "HIGH":
        return "HIGH: recommend human review"
    if risk_tier == "MEDIUM":
        return "MEDIUM: recommend step-up verification on refund"
    return "LOW: monitor — no immediate refund action"

# Add risk tier and action
case_df["community_size"] = case_df["account_id"].map(
    features_b.set_index("account_id")["community_size"].to_dict()
).fillna(0)
case_df["risk_tier"] = case_df.apply(lambda r: get_risk_tier(r["rank"], r["community_size"]), axis=1)
case_df["recommended_action"] = case_df["risk_tier"].apply(recommend_action)

# Generate case reports (simplified without detailed SHAP, but structure included)
case_reports = []
for _, row in case_df.iterrows():
    report = build_case_report(row, None)
    case_reports.append({"account_id": row["account_id"], "rank": row["rank"],
                         "proba": row["proba"], "case_report_text": report})
case_reports_df = pd.DataFrame(case_reports)
case_reports_df.to_csv(CASE_REPORTS_PATH, index=False)
print("Case reports saved.")

Case reports saved.


In [36]:
## Cell 16 — Bounded actions and audit log
actions_df = case_df[["account_id", "rank", "proba", "risk_tier", "recommended_action"]].copy()
actions_df.to_csv(BOUNDED_ACTIONS_PATH, index=False)

audit_timestamp = pd.Timestamp.now()
audit_df = actions_df.copy()
audit_df.insert(0, "timestamp", audit_timestamp)
audit_df.insert(2, "model_version", "Ensemble_LGBM_B_GNN")
audit_df["top_k_flag"] = True
audit_df["case_report_generated"] = audit_df["account_id"].isin(case_reports_df["account_id"])
audit_df.to_csv(AUDIT_LOG_PATH, index=False)
print("Bounded actions and audit log saved.")

Bounded actions and audit log saved.


In [37]:
## Cell 17 — Final validation of artifacts
expected_files = [
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
]
for path in expected_files:
    assert Path(path).exists(), f"Missing output: {path}"

assert len(flagged_df) == n_flagged
assert len(evidence_df) == n_flagged
assert len(graph_evidence_df) == n_flagged
assert len(case_reports_df) == n_flagged
assert len(actions_df) == n_flagged
assert len(audit_df) == n_flagged

print("All artifacts validated.")
print(f"Investigation queue: {n_flagged}")
print(f"Risk tier distribution:\n{actions_df['risk_tier'].value_counts()}")

All artifacts validated.
Investigation queue: 303
Risk tier distribution:
risk_tier
LOW         243
MEDIUM       40
HIGH         15
CRITICAL      5
Name: count, dtype: int64


In [39]:
## Cell 18 — ROC / PR curves and score distributions for all models
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

# Ensure test_pred_df exists (should be created in Cell 6)
if "test_pred_df" not in globals():
    raise NameError("test_pred_df is not defined. Please run Cell 6 first.")

probas = {
    "Model A": test_pred_df["proba_A"].values,
    "Model B": test_pred_df["proba_B"].values,
    "GNN": test_pred_df["proba_GNN"].values,
    "Ensemble": test_pred_df["proba_Ensemble"].values,
}
y = test_pred_df["true_label"].values

# ROC curves
plt.figure(figsize=(8, 6))
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y, p)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Test Set)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig(EXPLAINABILITY_DIR / "roc_curves_all.png", dpi=150, bbox_inches="tight")
plt.close()

# PR curves
plt.figure(figsize=(8, 6))
for name, p in probas.items():
    precision, recall, _ = precision_recall_curve(y, p)
    pr_auc = average_precision_score(y, p)
    plt.plot(recall, precision, label=f"{name} (PR-AUC = {pr_auc:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves (Test Set)")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.savefig(EXPLAINABILITY_DIR / "pr_curves_all.png", dpi=150, bbox_inches="tight")
plt.close()

# Score distributions
for name, p in probas.items():
    plt.figure(figsize=(8, 5))
    plt.hist(p[y == 0], bins=50, alpha=0.6, label="Normal", density=True)
    plt.hist(p[y == 1], bins=50, alpha=0.6, label="Fraud", density=True)
    plt.xlabel("Predicted Probability")
    plt.ylabel("Density")
    plt.title(f"Score Distribution: {name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(EXPLAINABILITY_DIR / f"score_dist_{name.replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
    plt.close()

print("Plots saved in", EXPLAINABILITY_DIR)

Plots saved in D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\v4_realistic_30k\processed\explainability


In [40]:
## Cell 19 — Display top investigation queue (ensemble)
display(
    case_df[
        ["account_id", "rank", "proba", "risk_tier", "recommended_action",
         "total_graph_links", "missing_evidence_count"]
    ]
    .sort_values("rank")
    .head(20)
)

,account_id,rank,proba,risk_tier,recommended_action,total_graph_links,missing_evidence_count
0,A025455,1,0.993601,CRITICAL,CRITICAL: recommend human review + soft-hold r...,8,NaN
1,A010538,2,0.993387,CRITICAL,CRITICAL: recommend human review + soft-hold r...,18,NaN
2,A011351,3,0.992184,CRITICAL,CRITICAL: recommend human review + soft-hold r...,3,NaN
3,A026494,4,0.991451,CRITICAL,CRITICAL: recommend human review + soft-hold r...,10,NaN
4,A022581,5,0.991214,CRITICAL,CRITICAL: recommend human review + soft-hold r...,6,NaN
5,A025761,6,0.989381,HIGH,HIGH: recommend human review,7,NaN
6,A006574,7,0.988847,HIGH,HIGH: recommend human review,1,NaN
7,A010391,8,0.988060,HIGH,HIGH: recommend human review,14,NaN
8,A021748,9,0.987179,HIGH,HIGH: recommend human review,5,NaN
9,A004963,10,0.986951,HIGH,HIGH: recommend human review,2,NaN
